In [ ]:
import sys
sys.path.append("..")

from src.spark_utils import load_config, get_spark, read_from_mysql, write_to_mysql
from src.dimensional_model import build_dim_date, parse_yymmdd_to_date, decode_birth_number
from pyspark.sql import functions as F

cfg = load_config()
spark = get_spark(cfg["jdbc_jar"])

In [ ]:
dim_date = build_dim_date(spark, "1993-01-01", "1998-12-31")
write_to_mysql(dim_date, "dim_date", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("dim_date done")

In [ ]:
district_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="district")
write_to_mysql(district_src, "dim_district", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("dim_district done")

In [ ]:
client_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="client")

dim_client = (decode_birth_number(client_src)
    .select("client_id", "district_id", "birth_date", "gender"))

write_to_mysql(dim_client, "dim_client", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("dim_client done")

In [ ]:
account_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="account")

dim_account = (parse_yymmdd_to_date(account_src, "date_created", "open_date")
    .select("account_id", "district_id", "frequency", "open_date"))

write_to_mysql(dim_account, "dim_account", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("dim_account done")

In [ ]:
disp_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="disp")

bridge = (disp_src
    .withColumnRenamed("type", "disposition_type")
    .select("client_id", "account_id", "disposition_type"))

write_to_mysql(bridge, "bridge_client_account", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("bridge_client_account done")

In [ ]:
trans_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="trans")

fact_transactions = (parse_yymmdd_to_date(trans_src, "date_trans", "full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .select("trans_id", "account_id", "date_key", "type", "operation", "k_symbol", "amount", "balance"))

write_to_mysql(fact_transactions, "fact_transactions", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("fact_transactions done")

In [ ]:
loan_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="loan")

fact_loans = (parse_yymmdd_to_date(loan_src, "date_granted", "full_date")
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .select("loan_id", "account_id", "date_key", "amount", "duration", "payments", "status"))

write_to_mysql(fact_loans, "fact_loans", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("fact_loans done")

In [ ]:
order_src = read_from_mysql(spark, cfg["jdbc_url"], cfg["db_user"], cfg["db_password"], dbtable="`order`")

fact_orders = order_src.select("order_id", "account_id", "bank_to", "account_to", "k_symbol", "amount")

write_to_mysql(fact_orders, "fact_orders", cfg["jdbc_url"], cfg["db_user"], cfg["db_password"])
print("fact_orders done")

In [ ]:
spark.stop()